In [ ]:
# ============================================================
# KAGGLE NOTEBOOK (SMOKE TEST ENABLED + BUG FIXES)
# ============================================================

CURRENT_SEED = 34
SMOKE_TEST = False
USE_GT_DESCRIPTIONS = False # always false if we dont want to  enhance 
USE_GT_BBOX = True

# ╔══════════════════════════════════════╗
# ║  CELL 1 — Install (NO CHANGE)        ║
# ╚══════════════════════════════════════╝
print("\n[STATUS] ---> Installing dependencies...")
!pip install open-clip-torch hnswlib ultralytics transformers accelerate -q
print("[STATUS] ---> Dependencies installed.")

# ╔══════════════════════════════════════╗
# ║  CELL 2 — Imports & Setup (NO CHANGE)║
# ╚══════════════════════════════════════╝
print("\n[STATUS] ---> Importing libraries and setting up device...")
import os, random, json, yaml, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import open_clip
import hnswlib
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[STATUS] ---> Device selected: {device}")
OUTPUT = "/kaggle/working"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CURRENT_SEED)
print(f"[STATUS] ---> Random seed locked to: {CURRENT_SEED}")

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Dataset paths & loading (NO CHANGE)            ║
# ╚══════════════════════════════════════════════════════════╝
print("\n[STATUS] ---> Hunting for dataset folders in Kaggle backend...")
BASE = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs and 'gallery' in dirs and 'query' in dirs:
        BASE = root
        break

if BASE is None:
    raise FileNotFoundError(" Cannot find the train/gallery/query folders!")

TRAIN_DIR   = f"{BASE}/train"
GALLERY_DIR = f"{BASE}/gallery"
QUERY_DIR   = f"{BASE}/query"

def load_split(split_dir):
    data = []
    for item_folder in sorted(Path(split_dir).iterdir()):
        if item_folder.is_dir():
            item_id = item_folder.name
            for img_file in sorted(item_folder.glob("*.jpg")):
                data.append((str(img_file), item_id))
    return data

print("[STATUS] ---> Loading image paths into memory...")
train_data   = load_split(TRAIN_DIR)
gallery_data = load_split(GALLERY_DIR)
query_data   = load_split(QUERY_DIR)

if SMOKE_TEST:
    print("\n SMOKE TEST IS ACTIVE! Truncating datasets to 64 images...")
    train_data   = train_data[:64]
    gallery_data = gallery_data[:64]
    query_data   = query_data[:64]

print(f" Loaded {len(train_data)} train, {len(gallery_data)} gallery, {len(query_data)} query images.")





# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3b — GT Bbox + Descriptions + Filter  [NEW CELL]   ║
# ╚══════════════════════════════════════════════════════════╝

# ── NEW: Set clothes-type filter ─────────────────────────────
# 1 = upper-body | 2 = lower-body | 3 = full-body | None = all
CLOTHES_TYPE_FILTER = None
CLOTHES_LABEL  = {1: "Upper-body", 2: "Lower-body", 3: "Full-body", None: "All"}
CLOTHES_CLASS  = {1: 0, 2: 1, 3: 2}   # clothes_type → YOLO class id

# ── NEW: Parse list_bbox_inshop.txt ──────────────────────────
# Gives us GT bounding boxes + clothes_type per image
# Used for: (a) YOLO fine-tune labels, (b) crop fallback, (c) split filtering
def parse_bbox_file(bbox_path):
    bbox_map = {}
    with open(bbox_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    for line in lines[2:]:      # line 0 = count, line 1 = header
        parts = line.split()
        if len(parts) < 7: continue
        p = Path(parts[0])      # img/WOMEN/.../id_00000001/02_1_front.jpg
        full_filename = f"{p.parts[1]}_{p.parts[2]}_{p.parts[3]}_{p.name}"

        bbox_map[(p.parent.name, full_filename)] = {
            "clothes_type": int(parts[1]),  # 1=upper 2=lower 3=full
            "pose_type":    int(parts[2]),
            "bbox":         (int(parts[3]), int(parts[4]),
                             int(parts[5]), int(parts[6])),
        }
    return bbox_map

bbox_file_path = None
for root, _, files in os.walk('/kaggle/input'):
    if "list_bbox_inshop.txt" in files:
        bbox_file_path = os.path.join(root, "list_bbox_inshop.txt"); break

if bbox_file_path is None:
    raise FileNotFoundError("list_bbox_inshop.txt not found!")
print(f"[STATUS] ---> Parsing GT bbox file...")
bbox_map = parse_bbox_file(bbox_file_path)
print(f" {len(bbox_map):,} GT bbox entries loaded.")

# ── NEW: Parse list_description_inshop.json ──────────────────
# Used for: enriching CLIP text embeddings alongside BLIP-2 captions
desc_file_path = None
for root, _, files in os.walk('/kaggle/input'):
    if "list_description_inshop.json" in files:
        desc_file_path = os.path.join(root, "list_description_inshop.json"); break

item_descriptions = {}
if desc_file_path is None:
    print("  list_description_inshop.json not found — GT descriptions unavailable.")
else:
    print(f"[STATUS] ---> Parsing GT descriptions...")
    with open(desc_file_path) as f:
        desc_data = json.load(f)
    for entry in desc_data:
        iid   = entry["item"]
        color = entry.get("color", "")
        lines = entry.get("description", [])
        # Color + first 2 lines gives concise but rich structured text
        text  = f"{color}. " + " ".join(lines[:2]) if lines else color
        item_descriptions[iid] = text.strip()
    print(f" {len(item_descriptions):,} GT descriptions loaded.")

def get_gt_description(item_id):
    # NEW: Returns GT description for item, or empty string if missing
    return item_descriptions.get(item_id, "")


# user can give which they want 
# ── NEW: Filter splits by clothes_type ───────────────────────
def filter_by_type(data, bmap, ctype):
    if ctype is None: return data
    return [(p, iid) for p, iid in data
            if bmap.get((iid, Path(p).name), {}).get("clothes_type") == ctype]

train_data   = filter_by_type(train_data,   bbox_map, CLOTHES_TYPE_FILTER)
gallery_data = filter_by_type(gallery_data, bbox_map, CLOTHES_TYPE_FILTER)
query_data   = filter_by_type(query_data,   bbox_map, CLOTHES_TYPE_FILTER)
print(f" Filter='{CLOTHES_LABEL[CLOTHES_TYPE_FILTER]}' → "
      f"train:{len(train_data):,} | gallery:{len(gallery_data):,} | query:{len(query_data):,}")





# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — YOLO fine-tuned on GT clothing bboxes [CHANGED]║
# ╚══════════════════════════════════════════════════════════╝

# ── CHANGED: Instead of loading pretrained YOLO and using it  ─
# as-is on persons, we fine-tune it on GT clothing bboxes      
# so it learns to detect clothing regions (3 classes:          
# upper-body, lower-body, full-body) rather than generic people

from ultralytics import YOLO

YOLO_TRAIN_DIR = f"{OUTPUT}/yolo_dataset"
YOLO_IMG_DIR   = f"{YOLO_TRAIN_DIR}/images/train"
YOLO_LBL_DIR   = f"{YOLO_TRAIN_DIR}/labels/train"
os.makedirs(YOLO_IMG_DIR, exist_ok=True)
os.makedirs(YOLO_LBL_DIR, exist_ok=True)

# ── CHANGED: Write YOLO label files from GT bboxes ───────────
print("[STATUS] ---> Writing YOLO label files from GT bboxes...")
yolo_img_count = 0
for img_path, item_id in tqdm(train_data, desc="Preparing YOLO labels"):
    filename = Path(img_path).name
    info     = bbox_map.get((item_id, filename))
    if info is None: continue
    try:
        img  = Image.open(img_path)
        W, H = img.size
    except: continue

    x1, y1, x2, y2 = info["bbox"]
    cx = ((x1 + x2) / 2) / W
    cy = ((y1 + y2) / 2) / H
    bw = (x2 - x1) / W
    bh = (y2 - y1) / H
    cx, cy = min(max(cx,0),1), min(max(cy,0),1)
    bw, bh = min(max(bw,0),1), min(max(bh,0),1)
    if bw < 0.01 or bh < 0.01: continue

    class_id = CLOTHES_CLASS[info["clothes_type"]]
    stem     = f"{item_id}_{Path(filename).stem}"
    dst_img  = f"{YOLO_IMG_DIR}/{stem}.jpg"
    if not os.path.exists(dst_img):
        shutil.copy2(img_path, dst_img)
    with open(f"{YOLO_LBL_DIR}/{stem}.txt", "w") as f:
        f.write(f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
    yolo_img_count += 1

print(f" {yolo_img_count:,} images labelled for YOLO fine-tuning.")

# ── CHANGED: Dataset YAML for YOLO ───────────────────────────
yolo_yaml_content = {
    "path":  YOLO_TRAIN_DIR,
    "train": "images/train",
    "val":   "images/train",
    "nc":    3,
    "names": ["upper-body", "lower-body", "full-body"],
}
yaml_path = f"{YOLO_TRAIN_DIR}/clothing.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yolo_yaml_content, f)


# ── CHANGED: Fine-tune YOLOv8n on clothing GT data ───────────
YOLO_EPOCHS = 1 if SMOKE_TEST else 10
YOLO_IMGSZ  = 320 if SMOKE_TEST else 640

print(f"[STATUS] ---> Fine-tuning YOLOv8n on clothing bboxes ({YOLO_EPOCHS} epochs)...")
yolo_base = YOLO("yolov8n.pt")   # start from pretrained weights
yolo_base.train(
    data     = yaml_path,
    epochs   = YOLO_EPOCHS,
    imgsz    = YOLO_IMGSZ,
    batch    = 16,
    project  = OUTPUT,
    name     = "yolo_clothing",
    exist_ok = True,
    verbose  = False,
    seed     = CURRENT_SEED,
)
YOLO_BEST = f"{OUTPUT}/yolo_clothing/weights/best.pt"
yolo_model = YOLO(YOLO_BEST)
print(f" Fine-tuned YOLO loaded from {YOLO_BEST}")






# ── CHANGED: crop_with_yolo now uses clothing-aware YOLO ──────
# Primary: fine-tuned YOLO detects clothing region
# Fallback 1: GT bbox (if YOLO finds nothing confident)
# Fallback 2: full image
def crop_with_yolo(img_path_or_pil, requested_type=None):

    YOLO_CLASS_MAP = {
        1: 0,   # upper-body
        2: 1,   # lower-body
        3: 2    # full-body
    }

    requested_yolo_class = YOLO_CLASS_MAP.get(requested_type)

    if isinstance(img_path_or_pil, str):
        img      = Image.open(img_path_or_pil).convert("RGB")
        item_id  = Path(img_path_or_pil).parent.name
        filename = Path(img_path_or_pil).name
        gt_info  = bbox_map.get((item_id, filename))
    else:
        img     = img_path_or_pil.convert("RGB")
        gt_info = None

    results = yolo_model(img, verbose=False)
    boxes   = results[0].boxes

    matching_boxes = [
        b for b in (boxes or [])
        if float(b.conf) > 0.4
        and (
            requested_yolo_class is None
            or int(b.cls[0]) == requested_yolo_class
        )
    ]

    if matching_boxes:

        best = max(matching_boxes, key=lambda b: float(b.conf))

        x1, y1, x2, y2 = map(int, best.xyxy[0].tolist())

        W, H = img.size

        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)

        if (x2-x1) >= 20 and (y2-y1) >= 20:
            return img.crop((x1, y1, x2, y2))

    # fallback GT bbox
    if USE_GT_BBOX and gt_info is not None:

        x1, y1, x2, y2 = gt_info["bbox"]

        W, H = img.size

        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)

        if (x2-x1) >= 20 and (y2-y1) >= 20:
            return img.crop((x1, y1, x2, y2))

    return img





# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Load CLIP (NO CHANGE)                          ║
# ╚══════════════════════════════════════════════════════════╝
print("\n[STATUS] ---> Loading Base CLIP Model (ViT-B-32)...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model = clip_model.to(device).eval()
tokenizer  = open_clip.get_tokenizer("ViT-B-32")
print(" CLIP loaded.")

def get_image_embedding(pil_image, model=None):
    m = model if model is not None else clip_model
    tensor = clip_preprocess(pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = m.encode_image(tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy().squeeze().astype("float32")

def get_text_embedding(text):
    tokens = tokenizer([text]).to(device)
    with torch.no_grad():
        emb = clip_model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy().squeeze().astype("float32")

def fuse(img_emb, txt_emb, alpha):
    f = alpha * img_emb + (1 - alpha) * txt_emb
    return (f / (np.linalg.norm(f) + 1e-9)).astype("float32")






# ── CHANGED: get_combined_text_embedding ─────────────────────
# BLIP-2 still generates the caption (its job in the pipeline).
# We append the GT description to give CLIP text encoder richer,
# structured metadata (color, material, fit) alongside the
# BLIP-2 visual description. Both together = better text embedding.
def get_combined_text_embedding(blip_caption, item_id=None):

    combined = blip_caption

    if USE_GT_DESCRIPTIONS and item_id is not None:
        gt_desc = get_gt_description(item_id)

        if gt_desc:
            combined = f"{blip_caption}. {gt_desc}"

    return get_text_embedding(combined)




# ── CHANGED: build_fused_index now takes item_ids ─────────────
# item_ids needed so we can look up GT description per gallery item if turned on
def build_fused_index(img_embeddings, captions, item_ids, alpha):
    fused = []
    for i, (cap, iid) in enumerate(zip(captions, item_ids)):
        txt_emb = get_combined_text_embedding(cap, iid)  # BLIP caption + GT desc
        fused.append(fuse(img_embeddings[i], txt_emb, alpha))
    fused = np.array(fused).astype("float32")
    return build_index(fused), fused

def build_index(embeddings):
    idx = hnswlib.Index(space="cosine", dim=embeddings.shape[1])
    idx.init_index(max_elements=len(embeddings), ef_construction=200, M=16)
    idx.add_items(embeddings)
    idx.set_ef(50)
    return idx







# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Embed gallery Config A (NO CHANGE IN LOGIC)    ║
# ╚══════════════════════════════════════════════════════════╝
# crop_with_yolo now uses fine-tuned YOLO — no code change needed here
print("\n[STATUS] ---> Starting Config A (Vision-Only) Gallery Embedding...")
gallery_emb_A, gallery_item_ids, gallery_img_paths = [], [], []
for img_path, item_id in tqdm(gallery_data, desc="Embedding gallery (Config A)"):
    try:
        emb = get_image_embedding(crop_with_yolo(img_path))
        gallery_emb_A.append(emb)
        gallery_item_ids.append(item_id)
        gallery_img_paths.append(img_path)
    except: pass

gallery_emb_A = np.array(gallery_emb_A).astype("float32")
print("[STATUS] ---> Building Config A HNSW Index...")
index_A = build_index(gallery_emb_A)
index_A.save_index(f"{OUTPUT}/index_A.bin")
print(" Config A processed and saved.")




# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Load BLIP-2 & Captions (BLIP IS FROZEN)    - NO CHANGE     ║
# ╚══════════════════════════════════════════════════════════╝
print("\n[STATUS] ---> Loading BLIP-2 Model (This takes ~60 seconds)...")
from transformers import Blip2Processor, Blip2ForConditionalGeneration
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16).to(device).eval()
print(" BLIP-2 loaded.")

print("\n[STATUS] ---> Generating captions for all gallery images...")
gallery_captions = []
for img_path, _ in tqdm(gallery_data, desc="Generating captions"):
    try:
        inputs = blip_processor(images=crop_with_yolo(img_path), return_tensors="pt").to(device, torch.float16)
        with torch.no_grad(): 
            out = blip_model.generate(**inputs, max_new_tokens=50)
        gallery_captions.append(blip_processor.decode(out[0], skip_special_tokens=True).strip())
    except: gallery_captions.append("a clothing item")


print("\n[STATUS] ---> Building Config B Fused Indices (alpha 0.7 and 0.5)...")
index_B_07, _ = build_fused_index(     gallery_emb_A,     gallery_captions, gallery_item_ids,     0.7 )
index_B_07.save_index(f"{OUTPUT}/index_B_07.bin")
index_B_05, _ = build_fused_index(gallery_emb_A, gallery_captions,gallery_item_ids, 0.5)
index_B_05.save_index(f"{OUTPUT}/index_B_05.bin")
print(" Config B processed and saved.")

print("[STATUS] ---> Clearing BLIP-2 from GPU memory to make room for training...")
import gc
del blip_model
gc.collect()
torch.cuda.empty_cache()



# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — CLIP Fine-tuning Config C (NO CHANGE)          ║
# ╚══════════════════════════════════════════════════════════╝
# crop_with_yolo is already updated — no changes needed here
print("\n[STATUS] ---> Setting up Fine-Tuning Pipeline...")
class FashionDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform
        all_ids = sorted(set(iid for _, iid in data))
        self.id_to_int = {iid: i for i, iid in enumerate(all_ids)}
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        path, item_id = self.data[idx]
        try: img = crop_with_yolo(path)
        except: img = Image.new("RGB", (224, 224))
        return self.transform(img), self.id_to_int[item_id]

class ImprovedInfoNCE(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temp = temperature
    def forward(self, embeddings, labels):
        emb  = embeddings / (embeddings.norm(dim=-1, keepdim=True) + 1e-9)
        sim  = torch.matmul(emb, emb.T) / self.temp
        lab  = labels.view(-1, 1)
        pos_mask = (lab == lab.T).float()
        pos_mask.fill_diagonal_(0.0)
        eye  = torch.eye(len(labels), device=emb.device).bool()
        sim  = sim - sim.max(dim=1, keepdim=True).values.detach()
        exp  = torch.exp(sim).masked_fill(eye, 0.0)
        log_prob  = sim - torch.log(exp.sum(dim=1, keepdim=True) + 1e-9)
        num_pos   = pos_mask.sum(dim=1).clamp(min=1.0)
        loss_per_item = -(log_prob * pos_mask).sum(dim=1) / num_pos
        return loss_per_item.mean()

clip_model_ft, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model_ft = clip_model_ft.to(device)
for param in clip_model_ft.visual.parameters(): param.requires_grad = False
for block in list(clip_model_ft.visual.transformer.resblocks)[-4:]:
    for param in block.parameters(): param.requires_grad = True

train_loader = DataLoader(FashionDataset(train_data, clip_preprocess),
                          batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
NUM_EPOCHS   = 1 if SMOKE_TEST else 10
WARMUP_EPOCHS = 1 if SMOKE_TEST else 2
criterion = ImprovedInfoNCE(temperature=0.07)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, clip_model_ft.parameters()),
                        lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[
    optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS),
    optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, NUM_EPOCHS - WARMUP_EPOCHS))
], milestones=[WARMUP_EPOCHS])

clip_model_ft.train()
print(f"\n[STATUS] ---> STARTING FINE-TUNING FOR SEED {CURRENT_SEED} ({NUM_EPOCHS} Epochs)...")
for epoch in range(NUM_EPOCHS):
    total_loss, n_batches = 0.0, 0
    for batch_imgs, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False):
        loss = criterion(clip_model_ft.encode_image(batch_imgs.to(device)), batch_labels.to(device))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(clip_model_ft.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item(); n_batches += 1
    scheduler.step()
    print(f" Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {total_loss / max(1, n_batches):.4f}")

torch.save(clip_model_ft.state_dict(), f"{OUTPUT}/clip_finetuned_{CURRENT_SEED}.pt")
print(f" Weights saved.")



# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Config C Index [CHANGED: pass gallery_item_ids]║
# ╚══════════════════════════════════════════════════════════╝
print("\n[STATUS] ---> Re-embedding gallery with Fine-Tuned Model (Config C)...")
clip_model_ft.eval()
gallery_emb_C = []
for img_path, _ in tqdm(gallery_data, desc="Re-embedding gallery (Config C)"):
    try: gallery_emb_C.append(get_image_embedding(crop_with_yolo(img_path), model=clip_model_ft))
    except: gallery_emb_C.append(np.zeros(512, dtype="float32"))
gallery_emb_C = np.array(gallery_emb_C).astype("float32")

# ── CHANGED: gallery_item_ids passed so GT descriptions enrich text embeddings
print("[STATUS] ---> Building Config C Fused Indices (alpha 0.7 and 0.5)...")
index_C_07, _ = build_fused_index(gallery_emb_C, gallery_captions, gallery_item_ids, 0.7)
index_C_07.save_index(f"{OUTPUT}/index_C_07_{CURRENT_SEED}.bin")
index_C_05, _ = build_fused_index(gallery_emb_C, gallery_captions, gallery_item_ids, 0.5)
index_C_05.save_index(f"{OUTPUT}/index_C_05_{CURRENT_SEED}.bin")
print(" Config C processed and saved.")












# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Metadata & Download (NO CHANGE)               ║
# ╚══════════════════════════════════════════════════════════╝
print("\n[STATUS] ---> Creating Metadata CSV...")
metadata_df = pd.DataFrame([
    {
        "item_id": iid,
        "relative_path": f"{iid}/{Path(p).name}",
        "caption": cap,
        "clothes_type": bbox_map.get(
            (iid, Path(p).name), {}
        ).get("clothes_type")
    }
    for p, iid, cap in zip(
        gallery_img_paths,
        gallery_item_ids,
        gallery_captions
    )
])
metadata_df.to_csv(f"{OUTPUT}/gallery_metadata.csv", index=False)
!zip -j -q /kaggle/working/files_seed_{CURRENT_SEED}.zip /kaggle/working/*.pt /kaggle/working/*.bin /kaggle/working/*.csv
from IPython.display import FileLink, display
print(f"\n ALL DONE FOR SEED {CURRENT_SEED}!")
display(FileLink(f"files_seed_{CURRENT_SEED}.zip"))


[STATUS] ---> Installing dependencies...
[STATUS] ---> Dependencies installed.

[STATUS] ---> Importing libraries and setting up device...
[STATUS] ---> Device selected: cuda
[STATUS] ---> Random seed locked to: 34

[STATUS] ---> Hunting for dataset folders in Kaggle backend...
[STATUS] ---> Loading image paths into memory...

 SMOKE TEST IS ACTIVE! Truncating datasets to 64 images...
 Loaded 64 train, 64 gallery, 64 query images.
[STATUS] ---> Parsing GT bbox file...
 52,712 GT bbox entries loaded.
[STATUS] ---> Parsing GT descriptions...
 7,982 GT descriptions loaded.
 Filter='All' → train:64 | gallery:64 | query:64
[STATUS] ---> Writing YOLO label files from GT bboxes...


Preparing YOLO labels:   0%|          | 0/64 [00:00<?, ?it/s]

 64 images labelled for YOLO fine-tuning.
[STATUS] ---> Fine-tuning YOLOv8n on clothing bboxes (1 epochs)...
Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/clothing.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, mult

Embedding gallery (Config A):   0%|          | 0/64 [00:00<?, ?it/s]

[STATUS] ---> Building Config A HNSW Index...
 Config A processed and saved.

[STATUS] ---> Loading BLIP-2 Model (This takes ~60 seconds)...


...

Epoch 1/1:   0%|          | 0/2 [00:00<?, ?it/s]

 Epoch 1/1 | Loss: 2.6440
 Weights saved.

[STATUS] ---> Re-embedding gallery with Fine-Tuned Model (Config C)...


Re-embedding gallery (Config C):   0%|          | 0/64 [00:00<?, ?it/s]

[STATUS] ---> Building Config C Fused Indices (alpha 0.7 and 0.5)...
 Config C processed and saved.

[STATUS] ---> Creating Metadata CSV...

 ALL DONE FOR SEED 34!


/kaggle/working/files_seed_34.zip